In [5]:
""" Get QoS """
import common_utils
import os
import pandas as pd
import difflib

from run_metadata import RunMetadata

# Example usage
metadata = RunMetadata()
root_folder = metadata.root_folder
filename = 'worker1.feather'
subfolders = common_utils.find_subfolders_with_file(root_folder, filename)
print(subfolders)
prom_data_paths = {os.path.basename(x): x for x in subfolders}
yolo_data_paths = {key: os.path.join(val, "worker_qos.feather") for key, val in prom_data_paths.items()}


['../../../data_warehouse/minimized_warehouse_7cc\\1738881900_(1.1000)', '../../../data_warehouse/minimized_warehouse_7cc\\1738915261_(1.5000)', '../../../data_warehouse/minimized_warehouse_7cc\\1738948257_(1.10000)']


In [6]:
import numpy as np


# Clean dataframe and calculate power
def get_mean_cpu(dataframe):
    cleaned_df = dataframe

    """ Sort by timestamp to make sure it makes sense to compute difference between first and last values """
    cleaned_df.sort_values(by="timestamp", inplace=True)

    """ Get all relevant columns """
    target_word = 'node load5'
    closest_matches = difflib.get_close_matches(target_word, cleaned_df.columns, n=1, cutoff=0.05)

    return cleaned_df[closest_matches[0]].mean()

mean_cpu_usages = {}
for key in prom_data_paths.keys():
    paths = []
    """ Get all workers """
    for work_num in range(1, 6):
        temp_path = os.path.join(prom_data_paths[key], f"worker{work_num}.feather")
        paths.append(temp_path)

    """ Get cpu per image for each worker """
    cpu_avg_per_worker = [get_mean_cpu(common_utils.get_cleaned_df(x)) for x in paths]
    cpu_avg = np.mean(cpu_avg_per_worker)

    """ Add result to dict for current model and resolution """
    model_info = common_utils.path_to_workers_and_pcl_size(key)
    if model_info.resolution not in mean_cpu_usages:
        mean_cpu_usages[model_info.resolution] = {}
    mean_cpu_usages[model_info.resolution][model_info.num_vehicles] = cpu_avg

max_cpu = {}
for resolution in sorted(mean_cpu_usages.keys()):
    cpu = pd.DataFrame.from_dict(mean_cpu_usages[resolution], orient='index', columns=['Joules'])
    cpu.columns = [f'{resolution}']
    max_cpu[resolution] = cpu



In [7]:
from matplotlib import pyplot as plt
# Grouped bars
import plotly.express as px
import numpy as np

# Define width based on resolution
# resolution_to_width = {160: 0.2, 320: 0.4, 640: 0.6, 1280: 0.8}
max_cpu_df = pd.concat(max_cpu.values(), axis=1)
max_cpu_df_sorted = max_cpu_df.sort_index()  # Sort by index first

# Create separator rows with NaN values
separator_row1 = pd.DataFrame(index=["..."], columns=max_cpu_df_sorted.columns, data=np.nan)
separator_row2 = pd.DataFrame(index=["...."], columns=max_cpu_df_sorted.columns,
                              data=np.nan)  # Using .... to make it unique

# Split the dataframe into three parts and insert the separators
mask1 = max_cpu_df_sorted.index <= 10
mask2 = (max_cpu_df_sorted.index > 10) & (max_cpu_df_sorted.index <= 20)
mask3 = max_cpu_df_sorted.index > 20

df_part1 = max_cpu_df_sorted[mask1]
df_part2 = max_cpu_df_sorted[mask2]
df_part3 = max_cpu_df_sorted[mask3]

# Combine all parts with the separators
max_cpu_df_sorted = pd.concat([df_part1, separator_row1, df_part2, separator_row2, df_part3])

# Convert remaining numeric indices to strings
max_cpu_df_sorted.index = max_cpu_df_sorted.index.astype(str)

fig = px.bar(max_cpu_df_sorted, barmode='group', title=f'{metadata.run_name}: Average CPU utilisation',
             labels={'value': 'Max Power (Watts)', 'index': 'Model'})
fig.update_layout(xaxis_title='Num_workers', yaxis_title='CPU utilization', legend_title_text='Resolution',
                  xaxis={'categoryorder': 'array', 'categoryarray': max_cpu_df_sorted.index})
fig.show()


In [8]:
# Store CPU usage per worker for each resolution
worker_cpu_usage = {}

# Print available keys to understand the naming pattern
print("Available keys:", list(prom_data_paths.keys()))
print("\nAvailable resolutions:", sorted(mean_cpu_usages.keys()))

# Find keys for 30 workers case for each resolution
for resolution in sorted(mean_cpu_usages.keys()):
    # Find the key that contains "30" and the current resolution
    try:
        matching_key = next(key for key in prom_data_paths.keys()
                          if "1" in key and str(resolution) in key)
        print(f"\nFound key for resolution {resolution}: {matching_key}")

        # Get data for each worker
        for work_num in range(1, 6):
            temp_path = os.path.join(prom_data_paths[matching_key], f"worker{work_num}.feather")
            cpu_usage = get_mean_cpu(common_utils.get_cleaned_df(temp_path))
            if resolution not in worker_cpu_usage:
                worker_cpu_usage[resolution] = []
            worker_cpu_usage[resolution].append(cpu_usage)
    except StopIteration:
        print(f"\nNo matching key found for resolution {resolution}")
        continue

# Create plots only for resolutions where we found data
for resolution in sorted(worker_cpu_usage.keys()):
    df = pd.DataFrame({
        'Worker': [f'Worker {i+1}' for i in range(5)],
        'CPU Usage': worker_cpu_usage[resolution]
    })

    fig = px.bar(df, x='Worker', y='CPU Usage',
                 title=f'CPU Utilization per Worker (Resolution: {resolution}, 30 workers)')
    fig.update_layout(yaxis_title='CPU Usage',
                     showlegend=False)
    fig.show()

Available keys: ['1738881900_(1.1000)', '1738915261_(1.5000)', '1738948257_(1.10000)']

Available resolutions: [1000, 5000, 10000]

Found key for resolution 1000: 1738881900_(1.1000)

Found key for resolution 5000: 1738915261_(1.5000)

Found key for resolution 10000: 1738948257_(1.10000)
